# Demo

This notebook is a demo of TransforMol LLM agent system, that analyzes the user inquiries and connects to the 4 specialised quantum chemistry chemistry ML models.


In [ ]:
import os
import sys

from transformol.agent_system import TransforMolAgentConfig
from transformol.agent_system import build_agent
from transformol.agent_system import TransforMolAgentConfig, build_agent

# Direct tool usage example
import json
from transformol.agent_system.tools.solv_deltag_tool import build_solv_deltag_tool
from transformol.agent_system.tools.reactive_atom_tool import build_reactive_atom_tool
from transformol.agent_system.tools.reaction_tool import build_reaction_tool
from transformol.agent_system.tools.solv_strc_tool import build_solv_strc_tool

from transformol.agent_system.mol_utils import (
    get_solvent_smiles,
    get_dielectric,
    resolve_solvent,
    atomic_number_to_symbol,
    symbol_to_atomic_number,
    xyz_text_to_arrays,
)

from pathlib import Path
REPO_ROOT = Path("../..").resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("Repository root:", REPO_ROOT)
print("Python path entry:", REPO_ROOT in [Path(p).resolve() for p in sys.path])

In [ ]:
# LLM provider: "anthropic (claude)", "openai (GPT-4o)", and "google (gemini)"
provider = "google"
# os.environ["ANTHROPIC_API_KEY"] = "XXX"
# os.environ["OPENAI_API_KEY"]    = "XXX"
os.environ["GOOGLE_API_KEY"]    = "XXX"

_KEY_MAP = {
    "anthropic": "ANTHROPIC_API_KEY",
    "openai":    "OPENAI_API_KEY",
    "google":    "GOOGLE_API_KEY",
}

key_var = _KEY_MAP.get(provider)
if key_var and not os.getenv(key_var):
    print(f"WARNING: {key_var} is not set!")
else:
    print(f"API key for '{provider}' ({key_var}) found")

### 1. Configure the Agent

In [ ]:
config = TransforMolAgentConfig(
    llm_provider=provider,
    # llm_model="gemini-2.0-flash",
    # llm_model="gemini-1.5-pro",
    # llm_model="claude-sonnet-4-5",
    # llm_model="gpt-4o",
    temperature=0.0,
    verbose=True,

    # Model checkpoint paths
    # Provide actual paths once you have trained models!
    # e.g. "transformol/solv_deltaG/best_model.pt"
    reaction_checkpoint=None,
    reactive_atom_checkpoint=None,
    solv_strc_checkpoint=None,
    solv_strc_metadata=None,
    device="cpu",
)

print(f"LLM provider : {config.llm_provider}")
print(f"LLM model    : {config.llm_model}")
print(f"Device       : {config.device}")

### 2. Build the Agent

In [ ]:
agent = build_agent(config)
print("Agent built successfully")
print(f"Tools: {[t.name for t in agent.tools]}")

### 3. Demo Queries

### 3.1 Solvation Free Energy

In [ ]:
query_1 = (
    "I have the SMILES 'CC' for ethane. "
    "Please predict its solvation Gibbs free energy in water."
)

print("Query:", query_1)
print("=" * 60)
result_1 = agent.invoke({"input": query_1})
print("\nAgent answer:\n", result_1["output"])

### 3.2 Reactive Atom Prediction

In [ ]:
query_2 = (
    "For the molecule cyclohexane (SMILES: C1CCCCC1), "
    "which atoms are most likely to be reactive at a transition state?"
)

print("Query:", query_2)
print("=" * 60)
result_2 = agent.invoke({"input": query_2})
print("\nAgent answer:\n", result_2["output"])

### 3.3 Reaction Prediction

In [ ]:
query_3 = (
    "Given the reactant molecule with SMILES CCO (ethanol), "
    "predict the reaction outcome and transition state."
)

print("Query:", query_3)
print("=" * 60)
result_3 = agent.invoke({"input": query_3})
print("\nAgent answer:\n", result_3["output"])

### 3.4 Solute structure in implicit solvent

In [ ]:
query_4 = (
    "I have the molecule ethanol (SMILES: CCO). "
    "Predict how its geometry changes when dissolved in DMSO solvent."
)

print("Query:", query_4)
print("=" * 60)
result_4 = agent.invoke({"input": query_4})
print("\nAgent answer:\n", result_4["output"])

### 4. Switching between LLM backends

You can swap backends without rebuilding tools - just create a new config and agent

In [ ]:
# ── Example: switch to Gemini 1.5 Pro for a single query ────────────────────

gemini_config = TransforMolAgentConfig(
    llm_provider="google",
    llm_model="gemini-2.0-flash",   # or "gemini-1.5-pro", "gemini-2.0-flash-lite"
    temperature=0.0,
)

# anthropic_config = TransforMolAgentConfig(llm_provider="anthropic")  # Claude
# openai_config    = TransforMolAgentConfig(llm_provider="openai")      # GPT-4o

print(f"Gemini model : {gemini_config.llm_model}")

# Build agent with Gemini
# gemini_agent = build_agent(gemini_config)
# result = gemini_agent.invoke({"input": 'Predict solvation Gibbs free energy of "CC" in water'})
# print(result["output"])

### 5. Using individual tools directly

You can also call each tool independently without going through the agent

Basically, you can go directly to the sub-directory of the ML expert tool that you want to use

In [ ]:
# Build tools
solv_deltag_tool   = build_solv_deltag_tool(config)
reactive_atom_tool = build_reactive_atom_tool(config)
reaction_tool      = build_reaction_tool(config)
solv_strc_tool     = build_solv_strc_tool(config)

print("All tools built!")

In [ ]:
# Direct solvation Gibbs free energybbs free energybbs free energybbs free energybbs free energybbs free energy prediction
result = solv_deltag_tool.run(json.dumps({"solute_smiles": "c1ccccc1", "solvent": "ethanol"}))
print(result)

In [ ]:
# Direct reactive atom prediction
result = reactive_atom_tool.run(json.dumps({"smiles": "c1ccccc1", "top_k": 3}))
print(result)

In [ ]:
# Direct reaction prediction
result = reaction_tool.run(json.dumps({"smiles": "CC", "num_samples": 2}))
print(result)

In [ ]:
# Direct solute structure prediction
result = solv_strc_tool.run(json.dumps({"smiles": "CCO", "solvent": "water"}))
print(result)

### 6. Molecular Utilities

The `mol_utils` module provides helpers used across all tools

In [ ]:
# Solvent name resolution
for name in ["water", "DMSO", "ethanol", "THF", "benzene"]:
    smiles = get_solvent_smiles(name)
    eps = get_dielectric(name)
    print(f"  {name:15s}  SMILES: {smiles or '(not found)':8s}   ε = {eps:.1f}")

In [ ]:
for z in [1, 6, 7, 8, 16, 17]:
    sym = atomic_number_to_symbol(z)
    z_back = symbol_to_atomic_number(sym)
    print(f"  Z={z:2d}  →  {sym}  →  Z={z_back}")

In [ ]:
xyz_example = """\
5
methane
C    0.000000  0.000000  0.000000
H    0.629118  0.629118  0.629118
H   -0.629118 -0.629118  0.629118
H   -0.629118  0.629118 -0.629118
H    0.629118 -0.629118 -0.629118
"""

atoms, coords = xyz_text_to_arrays(xyz_example)
print(atoms)
print(coords.shape)

### 7. How to add a trained model

Once you have trained a model (e.g. the solvation G free energy model), configure the checkpoint path

```python
config = TransforMolAgentConfig(
    llm_provider="google", # or "anthropic" / "openai"
    llm_model="gemini-2.0-flash", # optional override
    solv_deltag_checkpoint="/path/to/transformol/solv_deltaG/best_model.pt",
    solv_deltag_atom_dim=30, # must match training
    solv_deltag_bond_dim=6,
    device="cpu",
)
agent = build_agent(config)
result = agent.invoke({"input": 'Predict solvation Gibbs free energy of "CC" in water'})
print(result["output"])
```

Supported checkpoint formats:
- **Raw state_dict**: `torch.save(model.state_dict(), path)`
- **Dict with `model_state_dict`**: `{"model_state_dict": ..., "epoch": ...}`
- **Dict with `model_state`**: `{"model_state": ..., "epoch": ...}`